In [6]:
import re
import pandas as pd
import numpy as np
import json
import matplotlib.pyplot as plt

In [7]:
def clean_parse_file():
	input_file = "./steam-dataset/games.csv"
	output_file = "./steam-dataset/games_fixed.csv"



	pattern = re.compile(r',(?=[^{}]*\})') 

	with open(input_file, "r", encoding="utf-8") as infile, \
		open(output_file, "w", encoding="utf-8") as outfile:

		for line in infile:
			line = pattern.sub("|", line)
			outfile.write(line)


df_games = pd.read_csv("./steam-dataset/games_fixed_clean.csv", encoding="ISO-8859-1", na_values="\\N")


def see_nan_values(df):
	nan_values = df.isnull().sum()


df_games.dropna(subset=["price_overview"], inplace=True)

nan_values = df_games.isnull().sum()


s = df_games["price_overview"].astype("string")

df_games["price"] = pd.to_numeric(
	s.str.extract(r'final\\?"?\s*[:|]\s*(\d+(?:\.\d+)?)', expand=False)
) / 100

df_games["currency"] = s.str.extract(
	r'currency\\?"?\s*[:|]\s*\\?"?([A-Z]{3})',
	expand=False
)

df_games

df_games.drop(columns=["price_overview", "languages", "type"], inplace=True)


unique_curr = df_games['currency'].value_counts()



from currency_converter import CurrencyConverter
c = CurrencyConverter('./eurofxref-hist.csv')

df_games = df_games.replace('SAR', 'ZAR')

rates = {
	currency: c.convert(1, currency, 'EUR')
	for currency in c.currencies
}

rates['PEN'] = 0.26
rates['UAH'] = 0.019
rates['COP'] = 0.00028
rates['KWD'] = 2.78
rates['KZT'] = 0.0019
rates['TWD'] = 0.027
rates['AED'] = 0.23
rates['VND'] = 0.000033
df_games['prices_eur'] = df_games['price'] * df_games['currency'].map(rates)


df_games.drop(columns=["price", "currency", 'is_free'])

# df_games[df_games["prices_eur"] < df_games["prices_eur"].quantile(0.99)]['prices_eur'].hist()


def one_hot_encoding(df, column): 

	for category in df[column].unique():
		df[category] = df[column].apply(lambda x: 1 if category in x else 0)


	return df.drop(column, axis=1)

df_categories = pd.read_csv('./steam-dataset/categories.csv', na_values="\\N")
df_genres = pd.read_csv("./steam-dataset/genres.csv", na_values="\\N")
df_reviews = pd.read_csv("./steam-dataset/reviews.csv", na_values="\\N")
df_insights = pd.read_csv("./steam-dataset/steamspy_insights.csv", encoding="ISO-8859-1", na_values="\\N")
df_tags = pd.read_csv("./steam-dataset/tags.csv", na_values="\\N")



df_genres["genre"].value_counts()

df_reviews.dropna(thresh=1)
df_reviews_important = df_reviews.drop(['metacritic_score', 'reviews', 'recommendations', 'steamspy_user_score', 'steamspy_score_rank', 'steamspy_positive', 'steamspy_negative'], axis=1)

categories = df_reviews_important['review_score_description'].unique()
df_reviews_important.dropna(subset=['review_score_description'], inplace=True)
categories = df_reviews_important['review_score_description'].unique()


df_insights_clean = df_insights.drop(['developer', 'publisher', 'price', 'initial_price', 'discount', 'languages', 'genres', 'playtime_average_forever', 'playtime_average_2weeks', 'playtime_median_forever', 'playtime_median_2weeks'], axis=1)
df_insights_clean

df_tags_count = df_tags["tag"].value_counts()

df_games = df_games.drop(["price", "currency", "is_free"], axis=1)

games = df_games.merge(df_reviews_important.set_index("app_id"), on="app_id", how="inner")
games = games.merge(df_insights_clean.set_index("app_id"), on="app_id", how="inner")
games

def standardize_data(df):

	'''
	This function standardize an array, its substracts mean value,
	and then divide the standard deviation.

	param 1: array
	return: standardized array
	'''

	mean = df.mean()
	std = df.std()
	new_df = (df - mean) / std

	return new_df, mean, std

games_one_hot = one_hot_encoding(games, 'owners_range')
games_one_hot = one_hot_encoding(games_one_hot, 'review_score_description')

games_final = games_one_hot.drop(columns=['app_id', 'name', 'release_date'])
games_final = games_final.sample(frac=1)

In [13]:
df_x = games_final.iloc[:, :18]
df_y = games_final.iloc[:, 18:]

train_length = round(len(df_x) * .8)
df_x_train = df_x[:train_length]
df_x_test = df_x[train_length:]

df_y_train = df_y[:train_length]
df_y_test = df_y[train_length:]

df_x_standar, mean, std = standardize_data(df_x_train.iloc[:, :5])
df_x_train = pd.concat([df_x_standar.iloc[:, :], df_x_train.iloc[:, 5:]], axis=1)
df_x_train

df_x_test_standar = (df_x_test.iloc[:, :5] - mean) / std
df_x_test = pd.concat([df_x_test_standar, df_x_test.iloc[:, 5:]], axis=1)

display(df_x_train)
display(df_y_train)

,prices_eur,review_score,positive,negative,total,concurrent_users_yesterday,"10,000,000 .. 20,000,000","5,000,000 .. 10,000,000","2,000,000 .. 5,000,000","1,000,000 .. 2,000,000","500,000 .. 1,000,000","50,000,000 .. 100,000,000","20,000 .. 50,000","200,000 .. 500,000","100,000 .. 200,000","0 .. 20,000","50,000 .. 100,000","20,000,000 .. 50,000,000"
75152,-0.264463,0.868427,-0.066456,-0.068190,-0.068972,1,0,0,0,0,0,0,0,0,0,1,0,0
14915,-0.327635,1.171699,-0.055339,-0.065169,-0.058442,0,0,0,0,0,0,0,0,0,0,1,0,0
51184,-0.040561,-1.254480,-0.067023,-0.068190,-0.069491,0,0,0,0,0,0,0,0,0,0,1,0,0
74444,-0.344427,0.868427,-0.065718,-0.068190,-0.068298,3,0,0,0,0,0,0,0,0,0,1,0,0
9912,-0.424392,0.261882,-0.061975,-0.019846,-0.059064,0,0,0,0,0,0,0,0,1,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
38244,0.535185,0.565154,-0.060216,-0.049198,-0.060983,0,0,0,0,0,0,0,1,0,0,0,0,0
23723,2.054514,1.171699,-0.060500,-0.059989,-0.062540,2,0,0,0,0,0,0,0,0,0,1,0,0
58516,0.116169,-1.254480,-0.067136,-0.068190,-0.069594,0,0,0,0,0,0,0,0,0,0,1,0,0
9564,-0.136519,0.261882,-0.064867,-0.060421,-0.066586,0,0,0,0,0,0,0,0,0,0,1,0,0


,Overwhelmingly Positive,Very Positive,Mixed,Mostly Positive,No user reviews,Mostly Negative,Positive,Negative,Overwhelmingly Negative,Very Negative
75152,0,0,0,0,0,0,1,0,0,0
14915,0,1,0,0,0,0,1,0,0,0
51184,0,0,0,0,1,0,0,0,0,0
74444,0,0,0,0,0,0,1,0,0,0
9912,0,0,1,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...
38244,0,0,0,1,0,0,1,0,0,0
23723,0,1,0,0,0,0,1,0,0,0
58516,0,0,0,0,1,0,0,0,0,0
9564,0,0,1,0,0,0,0,0,0,0


In [ ]:
from sklearn.metrics import accuracy_score
from sklearn.ensemble import RandomForestClassifier

In [ ]:
rnd_clf = RandomForestClassifier(n_estimators=400, max_leaf_nodes=10, n_jobs=-1, random_state=42)
rnd_clf.fit(df_x_train, df_y_train)
y_pred_rf= rnd_clf.predict(df_x_test)
print("random forest", accuracy_score(df_y_test, y_pred_rf))
rnd_clf

In [ ]:
def capturar_datos_juego():
    print("=== CAPTURA DE DATOS DEL JUEGO DE STEAM ===")

    price_eur = float(input("Precio en EUR (ej. 19.99): "))
    review_score = float(
        input("Review score o Calificación (ej. 8 o 8.5): ")
    )
    positive = int(input("Cantidad de reseñas positivas (ej. 15000): "))
    negative = int(input("Cantidad de reseñas negativas (ej. 1200): "))
    total = int(
        input("Cantidad total de reseñas (ej. 16200): ")
        if positive is None
        else positive + negative
    )
    concurrent_users = int(
        input("Usuarios concurrentes ayer (ej. 4500): ")
    )

    rangos_owners = [
        "0 .. 20,000",
        "20,000 .. 50,000",
        "50,000 .. 100,000",
        "100,000 .. 200,000",
        "200,000 .. 500,000",
        "500,000 .. 1,000,000",
        "1,000,000 .. 2,000,000",
        "2,000,000 .. 5,000,000",
        "5,000,000 .. 10,000,000",
        "10,000,000 .. 20,000,000",
        "20,000,000 .. 50,000,000",
        "50,000,000 .. 100,000,000",
    ]
    grade = [
        "Overwhelmingly Positive",
        "Very Positive",
        "Mostly Positive",
        "Positive",
        "Mixed",
        "Negative",
        "Mostly Negative",	
        "Very Negative"
        "Overwhelmingly Negative",
        "No user reviews",
    ]

    print("\n--- Selecciona el rango de propietarios (Owners Range) ---")
    for i, rango in enumerate(rangos_owners, 1):
        print(f"[{i}] {rango}")

    # Validación de selección única
    while True:
        try:
            opcion = int(
                input(f"Elige una opción (1-{len(rangos_owners)}): ")
            )
            if 1 <= opcion <= len(rangos_owners):
                owners_range_selected = rangos_owners[opcion - 1]
                break
            else:
                print("Opción fuera de rango. Intenta de nuevo.")
        except ValueError:
            print("Por favor, ingresa solo un número entero.")
            
    # print("\n--- Selecciona la calificación de acuerdo a reseñas ---")
    # for i, rango in enumerate(grade, 1):
    #     print(f"[{i}] {rango}")

    # # Validación de selección única
    # while True:
    #     try:
    #         opcion = int(
    #             input(f"Elige una opción (1-{len(grade)}): ")
    #         )
    #         if 1 <= opcion <= len(grade):
    #             owners_range_selected = grade[opcion - 1]
    #             break
    #         else:
    #             print("Opción fuera de rango. Intenta de nuevo.")
    #     except ValueError:
    #         print("Por favor, ingresa solo un número entero.")

    # Guardar en un diccionario / DataFrame
    datos_capturados = {
        "price_eur": price_eur,
        "review_score": review_score,
        "positive": positive,
        "negative": negative,
        "total": total,
        "concurrent_users_yesterday": concurrent_users,
        "owners_range": owners_range_selected,
    }

    print("\n¡Datos capturados exitosamente!")
    return pd.DataFrame([datos_capturados])

In [15]:
capturar_datos_juego()

=== CAPTURA DE DATOS DEL JUEGO DE STEAM ===

--- Selecciona el rango de propietarios (Owners Range) ---
[1] 0 .. 20,000
[2] 20,000 .. 50,000
[3] 50,000 .. 100,000
[4] 100,000 .. 200,000
[5] 200,000 .. 500,000
[6] 500,000 .. 1,000,000
[7] 1,000,000 .. 2,000,000
[8] 2,000,000 .. 5,000,000
[9] 5,000,000 .. 10,000,000
[10] 10,000,000 .. 20,000,000
[11] 20,000,000 .. 50,000,000
[12] 50,000,000 .. 100,000,000

✓ ¡Datos capturados exitosamente!


,price_eur,review_score,positive,negative,total,concurrent_users_yesterday,owners_range
0,20.15,8.0,430000,1200,431200,4300,"50,000 .. 100,000"
